# Tasks for laboratory assignment 3

In [ ]:
# Imports
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, ExtraTreesRegressor


In [ ]:
def build_plot(file_path, x, y):
    data = pd.read_csv(file_path)
    plt.scatter(data[x], data[y], c='blue', marker='x')
    plt.xlabel(x)
    plt.ylabel(y)
    return plt

build_plot('resources/CO2_emission.csv', 'Engine_Size', 'CO2_Emissions').show()

In [ ]:
def extract_texts_from_file(file_path):
    """
    Load a CSV file and return it as a pandas DataFrame.
    The original notebook uses this helper for tabular ML datasets.
    """
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {file_path}. Put the required CSV in the resources folder."
        )
    return pd.read_csv(path)

data = extract_texts_from_file('resources/heart.csv')
print(data.head(50))


## CO2 emissions

Process file `CO2_emission.csv`. Build a model, that would predict CO2 emissions from the size of the engine of car. Visualize model on a plot.

Choose regression model that would best fit the data. Choose the library for model implementation (sklearn, other library, custom implementation).

In [ ]:
def CO2_emissions(data):
    """
    Fit a polynomial regression model predicting CO2 emissions from engine size.
    Returns [intercept, coefficient(s)] and plots observed data + fitted curve.
    """
    df = data.copy() if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    df.columns = [str(c).strip() for c in df.columns]

    def find_col(candidates):
        for candidate in candidates:
            for col in df.columns:
                if col.lower().replace(" ", "_") == candidate.lower().replace(" ", "_"):
                    return col
        for candidate in candidates:
            for col in df.columns:
                if candidate.lower() in col.lower():
                    return col
        raise ValueError(f"Could not identify column from {candidates}. Available: {list(df.columns)}")

    x_col = find_col(["Engine_Size", "engine size", "engine"])
    y_col = find_col(["CO2_Emissions", "CO2 emissions", "emissions"])
    clean = df[[x_col, y_col]].apply(pd.to_numeric, errors="coerce").dropna()
    X = clean[[x_col]].to_numpy()
    y = clean[y_col].to_numpy()

    # Quadratic polynomial typically captures the slight curvature in CO2 data.
    model = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
    model.fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    rmse = mean_squared_error(y, y_pred) ** 0.5

    grid = np.linspace(X.min(), X.max(), 250).reshape(-1, 1)
    grid_pred = model.predict(grid)
    plt.figure(figsize=(9, 5))
    plt.scatter(X[:, 0], y, alpha=0.65, label="Observed data")
    plt.plot(grid[:, 0], grid_pred, color="red", linewidth=2, label="Polynomial regression")
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f"CO2 emissions vs engine size (R²={r2:.3f}, RMSE={rmse:.2f})")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    Path("outputs").mkdir(exist_ok=True)
    plt.savefig("outputs/co2_emissions_regression.png", dpi=160)
    plt.show()

    linear = model.named_steps["linearregression"]
    return [float(linear.intercept_), *map(float, linear.coef_)]


## Ice Cream Sales

Process file `ice_cream_selling_data.csv`. Build a model, that would predict ice cream sales based on the temperature. Visualize model on a plot.

Choose regression model that would best fit the data. Choose the library for model implementation (sklearn, other library, custom implementation).

In [ ]:
def ice_cream_selling(data):
    """
    Predict ice-cream sales from temperature with a polynomial regression.
    Returns [intercept, coefficients] and plots the fitted curve.
    """
    df = data.copy() if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    df.columns = [str(c).strip() for c in df.columns]

    def find_col(candidates):
        for candidate in candidates:
            for col in df.columns:
                if col.lower().replace(" ", "_") == candidate.lower().replace(" ", "_"):
                    return col
        for candidate in candidates:
            for col in df.columns:
                if candidate.lower() in col.lower():
                    return col
        raise ValueError(f"Could not identify column from {candidates}. Available: {list(df.columns)}")

    x_col = find_col(["Temperature", "Temp"])
    y_col = find_col(["Ice Cream Sales", "Sales", "Revenue", "Units"])
    clean = df[[x_col, y_col]].apply(pd.to_numeric, errors="coerce").dropna()
    X, y = clean[[x_col]].to_numpy(), clean[y_col].to_numpy()

    model = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
    model.fit(X, y)
    pred = model.predict(X)
    r2 = r2_score(y, pred)
    rmse = mean_squared_error(y, pred) ** 0.5

    grid = np.linspace(X.min(), X.max(), 250).reshape(-1, 1)
    plt.figure(figsize=(9, 5))
    plt.scatter(X[:, 0], y, alpha=0.65, label="Observed sales")
    plt.plot(grid[:, 0], model.predict(grid), color="red", linewidth=2, label="Polynomial regression")
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f"Ice cream sales vs temperature (R²={r2:.3f}, RMSE={rmse:.2f})")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    Path("outputs").mkdir(exist_ok=True)
    plt.savefig("outputs/ice_cream_sales_regression.png", dpi=160)
    plt.show()

    linear = model.named_steps["linearregression"]
    return [float(linear.intercept_), *map(float, linear.coef_)]


## Electric Power Consumption

Process file `powerconsumption.csv`. Build a model, that would predict electricity consumption from the weather: temperature, humidity and wind speed. Visualize model on a plot.

Choose regression model that would best fit the data. Choose the library for model implementation (sklearn, other library, custom implementation).

In [ ]:
def power_consumption(data):
    """
    Predict power consumption from temperature, humidity and wind speed.
    Uses RandomForestRegressor and returns feature importances (theta-like values).
    """
    df = data.copy() if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    df.columns = [str(c).strip() for c in df.columns]
    lower = {c.lower().replace(" ", "").replace("_", ""): c for c in df.columns}

    def find_col(keys):
        for key in keys:
            key = key.lower().replace(" ", "").replace("_", "")
            if key in lower:
                return lower[key]
        for key in keys:
            key = key.lower().replace(" ", "").replace("_", "")
            for norm, col in lower.items():
                if key in norm:
                    return col
        raise ValueError(f"Could not identify column for {keys}. Available: {list(df.columns)}")

    temp = find_col(["temperature", "temp"])
    humidity = find_col(["humidity"])
    wind = find_col(["windspeed", "wind"])
    target = find_col(["powerconsumption", "power consumption", "consumption", "energy"])
    features = [temp, humidity, wind]
    clean = df[features + [target]].apply(pd.to_numeric, errors="coerce").dropna()
    X, y = clean[features], clean[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = RandomForestRegressor(n_estimators=250, random_state=42, min_samples_leaf=2)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(y_test, pred, alpha=0.65)
    bounds = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    axes[0].plot(bounds, bounds, "r--")
    axes[0].set_xlabel("Actual consumption")
    axes[0].set_ylabel("Predicted consumption")
    axes[0].set_title(f"Actual vs predicted (R²={r2:.3f})")
    axes[1].bar(features, model.feature_importances_)
    axes[1].set_ylabel("Importance")
    axes[1].set_title("Weather feature importance")
    axes[1].tick_params(axis="x", rotation=20)
    fig.suptitle(f"Power consumption prediction (RMSE={rmse:.2f})")
    fig.tight_layout()
    Path("outputs").mkdir(exist_ok=True)
    fig.savefig("outputs/power_consumption_model.png", dpi=160)
    plt.show()

    return model.feature_importances_.astype(float).tolist()


## Heart Conditions Predictions

Process file `heart.csv`. It has the following data:

- age - Age of the patient
- sex - Sex of the patient
- cp - Chest pain type ~ 0 = Typical Angina, 1 = Atypical Angina, 2 = Non-anginal Pain, 3 = Asymptomatic
- trtbps - Resting blood pressure (in mm Hg)
- chol - Cholestoral in mg/dl fetched via BMI sensor
- fbs - (fasting blood sugar > 120 mg/dl) ~ 1 = True, 0 = False
- restecg - Resting electrocardiographic results ~ 0 = Normal, 1 = ST-T wave normality, 2 = Left ventricular hypertrophy
- thalachh - Maximum heart rate achieved
- oldpeak - Previous peak
- slp - Slope
- caa - Number of major vessels
- thall - Thalium Stress Test result ~ (0,3)
- exng - Exercise induced angina ~ 1 = Yes, 0 = No
- output - Target variable - risk of heart attack - 1 is high chance of heart attack and 0 is low chance.

Use classification model (choose model best fitting model yourself). Choose parameters that would best fit for making a prediction on. Visualize model on a plot.

Do not forget to split data into train data and validation data. Validate your model.

In [ ]:
def heart_classification(data):
    """
    Train and validate a heart-risk classifier. Returns model coefficients
    when available; otherwise returns feature importances.
    """
    df = data.copy() if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    target_candidates = ["output", "target", "label", "heartdisease"]
    target = next((c for c in df.columns if c.lower() in target_candidates), None)
    if target is None:
        raise ValueError(f"Target column not found. Available columns: {list(df.columns)}")

    y = pd.to_numeric(df[target], errors="coerce")
    X = df.drop(columns=[target]).apply(pd.to_numeric, errors="coerce")
    valid = y.notna()
    X, y = X.loc[valid], y.loc[valid].astype(int)
    X = X.fillna(X.median(numeric_only=True))
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=42)
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"Validation accuracy: {accuracy_score(y_test, pred):.3f}")
    print(classification_report(y_test, pred, zero_division=0))
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot(cmap="Blues")
    plt.title("Heart risk classification — validation")
    plt.tight_layout()
    Path("outputs").mkdir(exist_ok=True)
    plt.savefig("outputs/heart_classification_confusion_matrix.png", dpi=160)
    plt.show()

    # Display validation probability distribution.
    if len(model.classes_) == 2:
        probabilities = model.predict_proba(X_test)[:, 1]
        plt.figure(figsize=(8, 4))
        plt.hist(probabilities, bins=15, edgecolor="black")
        plt.xlabel("Predicted probability of high risk")
        plt.ylabel("Number of patients")
        plt.title("Predicted heart-risk probabilities")
        plt.tight_layout()
        plt.savefig("outputs/heart_risk_probabilities.png", dpi=160)
        plt.show()

    return model.named_steps["logisticregression"].coef_[0].astype(float).tolist()


## Additional (extra credit)

There are more files in the resources folder. Take a look at the. Visualize data in these files. Choose ML algorithm for building a model and making predictions for new data. Visualize the model.

In [ ]:
def additional(data):
    """
    Generic exploratory ML for any tabular dataset: select a likely target,
    train a regression or classification model, print validation metrics,
    and visualize results. Returns the fitted estimator.
    """
    df = data.copy() if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
    if df.shape[1] < 2:
        raise ValueError("Additional dataset must contain at least two columns.")

    target = df.columns[-1]
    X = df.drop(columns=[target])
    y = df[target]
    X = pd.get_dummies(X, dummy_na=True)
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).fillna(0)

    if pd.api.types.is_numeric_dtype(y) and y.nunique() > 12:
        y = pd.to_numeric(y, errors="coerce")
        valid = y.notna()
        X, y = X.loc[valid], y.loc[valid]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        model = RandomForestRegressor(n_estimators=200, random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        print("Task: regression")
        print(f"R²: {r2_score(y_test, pred):.3f}")
        print(f"RMSE: {mean_squared_error(y_test, pred) ** 0.5:.3f}")
        plt.figure(figsize=(7, 5))
        plt.scatter(y_test, pred, alpha=0.6)
        plt.xlabel("Actual")
        plt.ylabel("Predicted")
        plt.title(f"Additional dataset: {target}")
    else:
        y = y.astype(str)
        stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=stratify
        )
        model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        print("Task: classification")
        print(f"Accuracy: {accuracy_score(y_test, pred):.3f}")
        print(classification_report(y_test, pred, zero_division=0))
        ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45)
        plt.title(f"Additional dataset: {target}")

    plt.tight_layout()
    Path("outputs").mkdir(exist_ok=True)
    plt.savefig("outputs/additional_dataset_model.png", dpi=160)
    plt.show()
    return model

# Automatically use the first additional CSV in resources, if one is present.
resource_dir = Path("resources")
main_files = {"co2_emission.csv", "heart.csv", "ice_cream_selling_data.csv", "powerconsumption.csv"}
extra_files = [f for f in resource_dir.glob("*.csv") if f.name.lower() not in main_files]
if extra_files:
    extra_path = extra_files[0]
    print(f"Using additional dataset: {extra_path}")
    additional(pd.read_csv(extra_path))
else:
    print("No additional CSV dataset found in resources; extra-credit task skipped.")


## Machine learning analysis on texts

Predict text score (sentiment, holistic score, etc., any other target label) using machine learning algorithm you see best fit for the task. Choose kaggle dataset from available below. Display classification report (confusion matrix, accuracy, f1 score, etc.). 

- Learning Agency Lab - Automated Essay Scoring 2.0: predict holistic text score given full_text (student essays) https://www.kaggle.com/competitions/learning-agency-lab-automated-essay-scoring-2
- 100K Coursera's Course Reviews Dataset: predict the final rating (1 to 5) based on the text of the review https://www.kaggle.com/datasets/septa97/100k-courseras-course-reviews-dataset
- LLM - Detect AI Generated Text: predict the target label (0 being written by student and 1 being written by LLM) given the full text https://www.kaggle.com/competitions/llm-detect-ai-generated-text
- 42K+ Udemy Course Enrollment Information Dataset🔥: classify course categories using Title and Summary; compare text-only features against combined title-and-summary features https://www.kaggle.com/datasets/songseungwon/2020-udemy-courses-dataset
- TMDB 5000 Movie Dataset: movie classification based on description https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata?select=tmdb_5000_movies.csv



In [ ]:
# Text classification using the selected Coursera reviews dataset.
# Kaggle authentication / internet access may be required to download it.
# If downloading does not work, place reviews.csv in resources/ and load it below.

def load_dataset(url, csv_file):
    """
    Load the requested Kaggle CSV with kagglehub.
    """
    import kagglehub
    from kagglehub import KaggleDatasetAdapter
    return kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, url, csv_file)

def classify_text_ml(df):
    """
    Predict review rating from review text using TF-IDF + LogisticRegression.
    Prints validation metrics and displays a confusion matrix.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import classification_report, accuracy_score, f1_score
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression

    data = df.copy()
    text_col = next((c for c in data.columns
                     if any(k in c.lower() for k in ["review", "text", "content", "comment"])), None)
    target_col = next((c for c in data.columns
                       if any(k in c.lower() for k in ["rating", "score", "label", "stars"])), None)
    if text_col is None or target_col is None:
        raise ValueError(f"Could not detect text and target columns. Columns: {list(data.columns)}")

    data = data[[text_col, target_col]].dropna()
    data[text_col] = data[text_col].astype(str)
    data[target_col] = data[target_col].astype(str)
    if len(data) > 30000:
        data = data.sample(30000, random_state=42)

    X_train, X_test, y_train, y_test = train_test_split(
        data[text_col], data[target_col], test_size=0.2, random_state=42,
        stratify=data[target_col]
    )
    model = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=40000, ngram_range=(1, 2),
                                  min_df=2, stop_words="english", sublinear_tf=True)),
        ("classifier", LogisticRegression(max_iter=1500, C=3.0))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
    print(f"Macro F1: {f1_score(y_test, pred, average='macro'):.4f}")
    print(classification_report(y_test, pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45)
    plt.title("Coursera review rating classification")
    plt.tight_layout()
    Path("outputs").mkdir(exist_ok=True)
    plt.savefig("outputs/text_classification_confusion_matrix.png", dpi=160)
    plt.show()
    return model

try:
    df = load_dataset('septa97/100k-courseras-course-reviews-dataset', 'reviews.csv')
except Exception as exc:
    print("Could not download the Kaggle dataset automatically.")
    print("Download reviews.csv from the Kaggle dataset and place it in resources/.")
    print(f"Details: {exc}")
    local_reviews = Path("resources/reviews.csv")
    df = pd.read_csv(local_reviews) if local_reviews.exists() else None

if df is not None:
    classify_text_ml(df)
